# VTuber Builder · Colab 기본 화면 (Gradio 없음)


## ① 저장소 준비 (AI 다운로드 없음)


In [ ]:
import os, pathlib, shutil, signal, subprocess, sys, time

REPO_URL = "https://github.com/jujumelona/Virtual-pipeline.git"
REPO_DIR = pathlib.Path("/content/Virtual-pipeline")
SETUP_LOG = pathlib.Path("/content/vtuber_builder/logs/colab_bootstrap.log")

def run(command, timeout):
    """Stream every stdout/stderr line in Colab and save the identical transcript."""
    SETUP_LOG.parent.mkdir(parents=True, exist_ok=True)
    label = subprocess.list2cmdline(command)
    print("[준비] $", label, "· 전체 로그:", SETUP_LOG, flush=True)
    with SETUP_LOG.open("a", encoding="utf-8") as output:
        output.write("\n$ " + label + "\n")
        output.flush()
        start_offset = output.tell()
        process = subprocess.Popen(
            command, stdout=output, stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    cursor = start_offset
    started = time.monotonic()

    def show_new_output():
        nonlocal cursor
        with SETUP_LOG.open("r", encoding="utf-8", errors="replace") as reader:
            reader.seek(cursor)
            while True:
                line = reader.readline()
                if not line:
                    break
                print(line, end="" if line.endswith("\n") else "\n", flush=True)
            cursor = reader.tell()

    try:
        while process.poll() is None:
            show_new_output()
            if time.monotonic() - started > timeout:
                raise TimeoutError(f"환경 준비 제한 시간 {timeout}초 초과 · {label}")
            time.sleep(0.1)
        show_new_output()
    except BaseException:
        if process.poll() is None:
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
            try:
                process.wait(timeout=3)
            except subprocess.TimeoutExpired:
                try:
                    os.killpg(process.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
                process.wait(timeout=10)
        show_new_output()
        raise
    if process.returncode:
        raise RuntimeError(
            f"환경 준비 실패(exit={process.returncode}): {label}\n"
            f"모든 stdout/stderr가 위에 출력되었습니다. 파일: {SETUP_LOG}"
        )

print("① 저장소 동기화")
if (REPO_DIR / ".git").is_dir():
    run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL], 60)
    run(["git", "-C", str(REPO_DIR), "fetch", "--prune", "origin", "main"], 180)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    run(["git", "clone", "--branch", "main", "--single-branch", REPO_URL, str(REPO_DIR)], 300)
run(["git", "-C", str(REPO_DIR), "checkout", "-B", "main", "origin/main"], 60)
run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"], 60)

# Force freshly checked out Python sources after a Colab ① re-run.
# The current kernel may still cache tools.colab_mode_ui from an older commit.
sys.path.insert(0, str(REPO_DIR))
import importlib
importlib.invalidate_caches()
_repo_root = REPO_DIR.resolve()
_stale_modules = []
for _mod_name, _mod in tuple(sys.modules.items()):
    _file = getattr(_mod, "__file__", None)
    if not _file:
        continue
    try:
        if pathlib.Path(_file).resolve().is_relative_to(_repo_root):
            _stale_modules.append(_mod_name)
    except (OSError, ValueError, TypeError):
        continue
for _mod_name in sorted(_stale_modules, key=lambda name: name.count("."), reverse=True):
    sys.modules.pop(_mod_name, None)
if _stale_modules:
    print("[① 버전 동기화] 오래된 모듈 캐시 교체:", len(_stale_modules), flush=True)

# Strict API handshake: fail here, not after the user has selected options.
from tools.colab_mode_ui import refresh_qwen, selection_signature, internal_scope, input_profile
from tools.colab_native import stop_legacy_server
stop_legacy_server()

print("① 저장소 연결 완료. ② 옵션 선택 후 ③ 다운로드 셀에서 프로그램과 모델을 병렬 준비하세요.", flush=True)


## ② 상위 제작 모드 선택

상위 모드부터 **실행 전에 선택**하세요. 아래 네 설정 셀 중 선택한 모드만 적용됩니다.


In [ ]:
#@title ② 상위 제작 모드 선택 — 실행 전에 선택
#@markdown **이 셀 위쪽의 Colab 기본 선택창에서 상위 모드 하나만 고르세요. 팝업이나 확인 버튼은 없습니다.**
#@markdown **Live2D** = Cubism/VTube Studio · **Inochi2D** = Inochi Creator/Session · **3D** = VRM · **액세서리** = 소품/교체 의상
TOP_MODE = "live2d" #@param ["live2d", "inochi2d", "3d", "accessory"]
if TOP_MODE not in ("live2d", "inochi2d", "3d", "accessory"):
    raise ValueError("상위 제작 모드가 올바르지 않습니다.")
TASK = "액세서리 제작" if TOP_MODE == "accessory" else "캐릭터 생성"
MODE = "3d" if TOP_MODE == "accessory" else TOP_MODE
# 미선택 모드의 설정 셀은 실행해도 자동으로 건너뜁니다.
LIVE2D_EDITION = "free"
LIVE2D_PRO_ASSET = "body"
LIVE2D_FRAMING = "upper"
LIVE2D_QWEN = "auto"
LIVE2D_USE_QWEN = False
USAGE = "personalNonProfit"
EXISTING_IMAGE_PATH = ""
MULTI_REFERENCE_3D = True
TWO_D_INPUT = "sheets"
ACCESSORY_SUBTYPE = "소품"
OUTFIT_2D_TARGET = "live2d"
ACCESSORY_ANCHOR = "AUTO"
ACCESSORY_BASE_VRM_PATH = ""
WARDROBE_2D_BASE_ZIP_PATH = ""
WARDROBE_XWEAR_PATH = ""
MODE_DETAILS_SELECTED = None
MODE_DETAILS_SNAPSHOT = None
MODEL_DOWNLOAD_READY = None
MODEL_DOWNLOAD_SELECTION = None
print("[② 상위 모드]", TOP_MODE, "— 해당 하위 설정 셀을 선택·실행하세요.", flush=True)


### ②-A Live2D 전용 설정


In [ ]:
#@title ③ Live2D 전용 하위 옵션 — Live2D 선택 시에만 적용
#@markdown **Live2D FREE/PRO 등급** · FREE=헤어·의상·장식 착용 완료 이미지 1장 · PRO=독립 신체·헤어·의상·액세서리 이미지
LIVE2D_EDITION_OPTION = "free" #@param ["free", "pro"]
#@markdown **PRO 독립 제작** · PRO일 때 선택한 한 종류만 제작. FREE일 때 무시.
LIVE2D_PRO_ASSET_OPTION = "body" #@param ["body", "hair", "outfit", "accessory"]
#@markdown **제작 범위** · upper=상반신(양손 포함), full=전신(양발 포함)
LIVE2D_FRAMING_OPTION = "upper" #@param ["upper", "full"]
#@markdown **Qwen 4bit + Stable-Layers** · auto=FREE 기본 분해/PRO 추가 Qwen, on=추가 실행, off=제외
LIVE2D_QWEN_OPTION = "auto" #@param ["auto", "on", "off"]
#@markdown **사용 범위** · 개인 비영리/개인 수익/기업 (외부 모델 라이선스 준수)
LIVE2D_USAGE_OPTION = "personalNonProfit" #@param ["personalNonProfit", "personalProfit", "corporation"]
#@markdown **기존 완성 원본 이미지 경로**(선택) · 비우면 이후 업로드
LIVE2D_EXISTING_IMAGE_OPTION = "" #@param {type:"string"}
if globals().get("TOP_MODE") == "live2d":
    from tools.colab_mode_ui import refresh_qwen, selection_signature
    LIVE2D_EDITION = LIVE2D_EDITION_OPTION
    LIVE2D_PRO_ASSET = LIVE2D_PRO_ASSET_OPTION
    LIVE2D_FRAMING = LIVE2D_FRAMING_OPTION
    LIVE2D_QWEN = LIVE2D_QWEN_OPTION
    USAGE = LIVE2D_USAGE_OPTION
    EXISTING_IMAGE_PATH = LIVE2D_EXISTING_IMAGE_OPTION
    refresh_qwen(globals())
    MODE_DETAILS_SNAPSHOT = selection_signature(globals())
    MODE_DETAILS_SELECTED = TOP_MODE
    print("[③ Live2D 설정 완료]", MODE_DETAILS_SNAPSHOT, flush=True)
else:
    print("[③ Live2D 설정 생략] 상위 모드:", globals().get("TOP_MODE"), flush=True)


### ②-B Inochi2D 전용 설정


In [ ]:
#@title ④ Inochi2D 전용 하위 옵션 — Inochi2D 선택 시에만 적용
#@markdown **입력 방식** · sheets=기준 이미지+시트 ZIP · provided_layers=분리 파츠 PNG · automatic=원본 자동 분리
INOCHI_INPUT_OPTION = "sheets" #@param ["sheets", "provided_layers", "automatic"]
#@markdown **사용 범위**
INOCHI_USAGE_OPTION = "personalNonProfit" #@param ["personalNonProfit", "personalProfit", "corporation"]
#@markdown **기존 원본 이미지 경로**(선택)
INOCHI_EXISTING_IMAGE_OPTION = "" #@param {type:"string"}
if globals().get("TOP_MODE") == "inochi2d":
    from tools.colab_mode_ui import selection_signature
    TWO_D_INPUT = INOCHI_INPUT_OPTION
    USAGE = INOCHI_USAGE_OPTION
    EXISTING_IMAGE_PATH = INOCHI_EXISTING_IMAGE_OPTION
    MODE_DETAILS_SNAPSHOT = selection_signature(globals())
    MODE_DETAILS_SELECTED = TOP_MODE
    print("[④ Inochi2D 설정 완료]", MODE_DETAILS_SNAPSHOT, flush=True)
else:
    print("[④ Inochi2D 설정 생략] 상위 모드:", globals().get("TOP_MODE"), flush=True)


### ②-C 3D VRM 전용 설정


In [ ]:
#@title ⑤ 3D VRM 전용 하위 옵션 — 3D 선택 시에만 적용
#@markdown **다중 시점 시트 사용** · 체크=정면·후면·좌우·얼굴 시트 ZIP · 해제=원본 이미지 1장
THREE_D_MULTIVIEW_OPTION = True #@param {type:"boolean"}
#@markdown **사용 범위**
THREE_D_USAGE_OPTION = "personalNonProfit" #@param ["personalNonProfit", "personalProfit", "corporation"]
#@markdown **기존 원본 이미지 경로**(선택)
THREE_D_EXISTING_IMAGE_OPTION = "" #@param {type:"string"}
if globals().get("TOP_MODE") == "3d":
    from tools.colab_mode_ui import selection_signature
    MULTI_REFERENCE_3D = THREE_D_MULTIVIEW_OPTION
    USAGE = THREE_D_USAGE_OPTION
    EXISTING_IMAGE_PATH = THREE_D_EXISTING_IMAGE_OPTION
    MODE_DETAILS_SNAPSHOT = selection_signature(globals())
    MODE_DETAILS_SELECTED = TOP_MODE
    print("[⑤ 3D VRM 설정 완료]", MODE_DETAILS_SNAPSHOT, flush=True)
else:
    print("[⑤ 3D VRM 설정 생략] 상위 모드:", globals().get("TOP_MODE"), flush=True)


### ②-D 액세서리·교체 의상 전용 설정


In [ ]:
#@title ⑥ 액세서리·교체 의상 전용 하위 옵션 — 액세서리 선택 시에만 적용
#@markdown **액세서리 제작 종류** · 소품 / 2D 교체 의상 / 3D XWear 의상
ACCESSORY_SUBTYPE_OPTION = "소품" #@param ["소품", "2D 교체 의상", "3D 교체 의상(XWear)"]
#@markdown **2D 교체 의상 대상** · 소품이나 XWear 모드에서는 무시
ACCESSORY_2D_TARGET_OPTION = "live2d" #@param ["live2d", "inochi2d"]
#@markdown **소품 부착 위치** · AUTO=파일명에서 판별, ALL=모든 위치
ACCESSORY_ANCHOR_OPTION = "AUTO" #@param ["AUTO", "ALL", "HEAD_TOP", "FACE", "LEFT_EAR", "RIGHT_EAR", "NECK", "CHEST", "BACK", "LEFT_SHOULDER", "RIGHT_SHOULDER", "LEFT_HAND", "RIGHT_HAND", "LEFT_FOOT", "RIGHT_FOOT", "HIPS"]
#@markdown **사용 범위**
ACCESSORY_USAGE_OPTION = "personalNonProfit" #@param ["personalNonProfit", "personalProfit", "corporation"]
#@markdown **기준 VRM 파일 경로**(선택)
ACCESSORY_BASE_VRM_OPTION = "" #@param {type:"string"}
#@markdown **2D 교체 의상 기준 ZIP 경로**(선택)
ACCESSORY_2D_BASE_ZIP_OPTION = "" #@param {type:"string"}
#@markdown **3D XWear 의상 파일 경로**(선택)
ACCESSORY_XWEAR_OPTION = "" #@param {type:"string"}
if globals().get("TOP_MODE") == "accessory":
    from tools.colab_mode_ui import selection_signature
    ACCESSORY_SUBTYPE = ACCESSORY_SUBTYPE_OPTION
    OUTFIT_2D_TARGET = ACCESSORY_2D_TARGET_OPTION
    ACCESSORY_ANCHOR = ACCESSORY_ANCHOR_OPTION
    USAGE = ACCESSORY_USAGE_OPTION
    ACCESSORY_BASE_VRM_PATH = ACCESSORY_BASE_VRM_OPTION
    WARDROBE_2D_BASE_ZIP_PATH = ACCESSORY_2D_BASE_ZIP_OPTION
    WARDROBE_XWEAR_PATH = ACCESSORY_XWEAR_OPTION
    MODE_DETAILS_SNAPSHOT = selection_signature(globals())
    MODE_DETAILS_SELECTED = TOP_MODE
    print("[⑥ 액세서리 설정 완료]", MODE_DETAILS_SNAPSHOT, flush=True)
else:
    print("[⑥ 액세서리 설정 생략] 상위 모드:", globals().get("TOP_MODE"), flush=True)


## ⑦ AI 모델·의존성·프로그램 다운로드


In [ ]:
#@title ⑦ AI 모델·의존성·필수 프로그램 다운로드 (독립 작업 병렬 실행)
#@markdown 2D/3D 이미지 생성 작업에 필요한 모델만 준비합니다. 3D XWear 의상은 VRoid 편집기에서 피팅하므로 GPU 모델 다운로드가 필요하지 않습니다.
import sys, pathlib
if "TASK" not in globals():
    raise RuntimeError("② 제작 옵션 셀을 먼저 실행하세요.")
from tools.colab_mode_ui import selection_signature
if globals().get("MODE_DETAILS_SELECTED") != globals().get("TOP_MODE"):
    raise RuntimeError("② 상위 모드에 해당하는 ③~⑥ 하위 설정 셀을 먼저 실행하세요.")
SELECTED_MODE = selection_signature(globals())
if globals().get("MODE_DETAILS_SNAPSHOT") != SELECTED_MODE:
    raise RuntimeError("하위 설정이 변경됐습니다. 선택한 ③~⑥ 설정 셀을 다시 실행하세요.")
repo = pathlib.Path("/content/Virtual-pipeline")
if not (repo / "tools" / "colab_mode_prepare.py").is_file():
    raise RuntimeError("① 저장소 설치/동기화 셀을 먼저 실행하세요.")
from tools.colab_mode_ui import internal_scope, input_profile
wardrobe_2d = TASK == "액세서리 제작" and ACCESSORY_SUBTYPE == "2D 교체 의상"
wardrobe_3d = TASK == "액세서리 제작" and ACCESSORY_SUBTYPE == "3D 교체 의상(XWear)"
scope = internal_scope(TASK, MODE, LIVE2D_EDITION, ACCESSORY_SUBTYPE, OUTFIT_2D_TARGET)
profile = input_profile(TASK, MODE, ACCESSORY_SUBTYPE, TWO_D_INPUT, MULTI_REFERENCE_3D)
print(f"[준비] 작업={TASK} 세부={ACCESSORY_SUBTYPE if TASK=='액세서리 제작' else MODE} · scope={scope}",flush=True)
if scope in ("live2d_free", "live2d_pro"):
    print("[VTS] 공통 2D 얼굴 랜드마크 모델과 의존성 먼저 준비",flush=True)
    run([sys.executable,"-u",str(repo/"tools"/"colab_mode_prepare.py"),
         "--mode","live2d","--usage",USAGE,
         "--provided-2d-layers","--no-first-gpu-prewarm"],timeout=7200)
    command = [sys.executable, "-u", str(repo/"tools"/"vts_setup.py")]
    if LIVE2D_USE_QWEN:
        command.append("--with-qwen")
    run(command, timeout=14400)
    print("[VTS] See-through NF4 설치·가중치 준비 완료. 실제 GPU 추론은 ⑤에서 검증.",flush=True)
elif wardrobe_3d:
    print("[준비] VRoid XWear는 외부 편집기 원본입니다. "
          "VRM/XWear 입력 검증만 실행하고 GPU를 설치하지 않습니다.",flush=True)
else:
    prewarm_face = (
        TASK == "캐릭터 생성"
        and not (MODE == "3d" and MULTI_REFERENCE_3D)
        and not (MODE in ("inochi2d","live2d") and TWO_D_INPUT == "sheets")
    )
    command = [sys.executable,"-u",str(repo/"tools"/"colab_mode_prepare.py"),
               "--mode",scope,"--usage",USAGE]
    if not prewarm_face:
        command.append("--no-first-gpu-prewarm")
    if (scope in ("live2d","inochi2d")
            and (profile in ("provided_layers","sheets","wardrobe_2d"))):
        command.append("--provided-2d-layers")
    if profile in ("sheets","wardrobe_2d"):
        command.append("--sheet-pack")
    run(command,timeout=12000)
MODEL_DOWNLOAD_READY = scope
MODEL_DOWNLOAD_SELECTION = SELECTED_MODE
MODEL_DOWNLOAD_QWEN = LIVE2D_USE_QWEN if scope in ("live2d_free","live2d_pro") else None
MODEL_DOWNLOAD_EDITION = LIVE2D_EDITION if MODE == "live2d" and TASK == "캐릭터 생성" else None
MODEL_DOWNLOAD_PROFILE = profile
print("⑦ 준비 및 검증 완료. ⑧ 업로드 셀을 실행하세요.",flush=True)


## ⑧ 사진·VRM 업로드


In [ ]:
#@title ⑧ 사진·액세서리 업로드 전용 셀 (제작하지 않음)
#@markdown 업로드는 이 셀에서만 실행합니다. ⑨ 제작 셀은 업로드 대화상자를 띄우지 않습니다.
from pathlib import Path
import sys, uuid
repo = Path("/content/Virtual-pipeline")
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
from tools.colab_native import _stored_uploads, _existing_image, _latest_avatar, _has_vrm_container
from tools.colab_mode_ui import selection_signature
if globals().get("MODEL_DOWNLOAD_SELECTION") != selection_signature(globals()):
    raise RuntimeError("② 옵션이 ⑦ 모델 준비 이후 바뀌었거나 ③을 실행하지 않았습니다. ⑦부터 다시 실행하세요.")
from google.colab import files

UPLOAD_ROOT = Path("/content/vtuber_builder/notebook_inputs") / ("manual_" + uuid.uuid4().hex)
INPUT_IMAGE_PATH = ""
PRO_BASE_REFERENCE_PATH = ""
REFERENCE_FACE_PATH = ""
REFERENCE_BACK_PATH = ""
REFERENCE_LEFT_PATH = ""
REFERENCE_RIGHT_PATH = ""
LAYER_ZIP_PATH = ""
SHEET_PACK_PATH = ""
UPLOADED_ACCESSORY_BASE = ""
UPLOADED_ACCESSORY_IMAGES = []
WARDROBE_BASE_SHEET_ZIP = ""
WARDROBE_OUTFIT_PNG = ""
WARDROBE_HAIR_PNG = ""
WARDROBE_BASE_VRM = ""
WARDROBE_XWEAR = ""

if TASK == "캐릭터 생성" and MODE == "live2d":
    root = UPLOAD_ROOT / "live2d"
    root.mkdir(parents=True, exist_ok=True)
    if LIVE2D_EDITION == "free":
        if EXISTING_IMAGE_PATH.strip():
            INPUT_IMAGE_PATH = _existing_image(EXISTING_IMAGE_PATH.strip(), description="Live2D FREE 완성 캐릭터")
        else:
            print("[Live2D FREE] 헤어·의상·액세서리를 착용한 완성 캐릭터 1장을 업로드하세요.", flush=True)
            INPUT_IMAGE_PATH = _stored_uploads(files.upload, root, images=True, multiple=False)[0]
        print("[Live2D FREE] 입력:", INPUT_IMAGE_PATH, flush=True)
    else:
        # PRO is an independent job: body, hair, outfit OR accessory.
        # Do not request every detachable part at once.
        kind = LIVE2D_PRO_ASSET
        filenames = {"body": "base_master", "hair": "hair_variant",
                     "outfit": "outfit_variant", "accessory": "accessories_variant"}
        selected_name = f"pro_{LIVE2D_FRAMING}_{filenames[kind]}.png"
        reference_name = f"pro_{LIVE2D_FRAMING}_base_master.png"
        required = {selected_name} if kind == "body" else {selected_name, reference_name}
        print("[Live2D PRO]", kind, "전용 제작. 필요한 파일:", sorted(required), flush=True)
        raw = files.upload()
        if set(raw) != required:
            raise ValueError("선택한 PRO 하위 모드의 파일만 올리세요. 필요="
                             + str(sorted(required)) + ", 관측=" + str(sorted(raw)))
        from PIL import Image
        from io import BytesIO
        dimensions = set()
        for name, payload in raw.items():
            if len(payload) > 128 * 1024 * 1024:
                raise ValueError(name + ": 파일 제한 128MiB 초과")
            with Image.open(BytesIO(payload)) as im:
                if im.format != "PNG":
                    raise ValueError(name + ": PNG 필수")
                im.load()
                dimensions.add(im.size)
            (root / name).write_bytes(payload)
        if len(dimensions) != 1:
            raise ValueError("PRO 자산과 신체 기준 이미지의 픽셀 좌표·크기가 다릅니다.")
        INPUT_IMAGE_PATH = str(root / selected_name)
        if kind != "body":
            PRO_BASE_REFERENCE_PATH = str(root / reference_name)
    LIVE2D_INPUT_MODE = MODE
    LIVE2D_INPUT_EDITION = LIVE2D_EDITION
    LIVE2D_INPUT_ASSET = LIVE2D_PRO_ASSET if LIVE2D_EDITION == 'pro' else None
    LIVE2D_INPUT_FRAMING = LIVE2D_FRAMING
    SHEET_PACK_PATH = ""
elif TASK == "캐릭터 생성" and (
    (MODE == "inochi2d" and TWO_D_INPUT == "sheets")
    or (MODE == "3d" and MULTI_REFERENCE_3D)
):
    from tools.sheet_input_loader import store_uploaded_zip
    name = ("character_3d_sheet_pack.zip" if MODE == "3d"
            else "character_2d_sheet_pack.zip")
    print("[업로드] README 규격의 " + name + " 한 개를 선택하세요.",flush=True)
    SHEET_PACK_PATH = store_uploaded_zip(
        files.upload(),str(UPLOAD_ROOT / "sheets"),MODE
    )
    print("[업로드] 모든 이미지 크기·격자·알파 검증 완료",flush=True)
elif TASK == "캐릭터 생성" and MODE == "inochi2d" and TWO_D_INPUT == "provided_layers":
    from tools.colab_input_images import prepare_2d_image_uploads
    print("[업로드] README의 파일명대로 생성한 PNG 이미지 21장(기준 1 + 중립 파츠 20)을 한 번에 선택하세요.", flush=True)
    print("[업로드] ZIP·프롬프트·색상 입력은 필요하지 않습니다. 이미지 파일만 선택합니다.", flush=True)
    UPLOAD_ROOT.mkdir(parents=True, exist_ok=True)
    INPUT_IMAGE_PATH, LAYER_ZIP_PATH = prepare_2d_image_uploads(
        files.upload(), str(UPLOAD_ROOT / "user_png_layers")
    )
    print("[업로드 확인] 원본 1장 + 의상·헤어 없는 투명 파츠 20장 (내부 제작 자료 자동 구성)", flush=True)
elif TASK == "캐릭터 생성":
    if EXISTING_IMAGE_PATH.strip():
        INPUT_IMAGE_PATH = _existing_image(EXISTING_IMAGE_PATH.strip(), description="캐릭터 원본")
        print("[업로드] 기존 캐릭터 파일 사용:", INPUT_IMAGE_PATH, flush=True)
    else:
        print("[업로드] 캐릭터 원본 이미지 1장을 선택하세요.", flush=True)
        INPUT_IMAGE_PATH = _stored_uploads(
            files.upload, UPLOAD_ROOT / "character", images=True, multiple=False
        )[0]
    print("[업로드 완료] 캐릭터:", INPUT_IMAGE_PATH, flush=True)
elif TASK == "액세서리 제작" and ACCESSORY_SUBTYPE == "2D 교체 의상":
    from tools.sheet_input_loader import inspect_sheet_archive, store_uploaded_zip
    if WARDROBE_2D_BASE_ZIP_PATH.strip():
        WARDROBE_BASE_SHEET_ZIP = str(Path(WARDROBE_2D_BASE_ZIP_PATH.strip()).expanduser().resolve(strict=True))
        inspect_sheet_archive(WARDROBE_BASE_SHEET_ZIP,"live2d")
    else:
        print("[2D 의상] 기존 캐릭터의 character_2d_sheet_pack.zip 파일을 업로드하세요.",flush=True)
        WARDROBE_BASE_SHEET_ZIP = store_uploaded_zip(
            files.upload(),str(UPLOAD_ROOT/"base_sheets"),"live2d")
    print("[2D 헤어·의상] outfit_variant.png는 필수, hair_variant.png는 선택사항입니다. "
          "파일 두 개를 함께 선택하면 머리+의상 전체 리깅 입력을 생성합니다.",flush=True)
    item = files.upload()
    expected = {"outfit_variant.png","hair_variant.png"}
    if "outfit_variant.png" not in item or not set(item).issubset(expected):
        raise ValueError("outfit_variant.png 필수; hair_variant.png 선택(정확한 파일명).")
    from tools.modular_avatar_pack import inspect_image
    dest = UPLOAD_ROOT/"wardrobe"
    dest.mkdir(parents=True,exist_ok=True)
    WARDROBE_OUTFIT_PNG = str(dest/"outfit_variant.png")
    Path(WARDROBE_OUTFIT_PNG).write_bytes(item["outfit_variant.png"])
    print("[2D 의상] 유효한 분리형 파츠:",inspect_image(WARDROBE_OUTFIT_PNG,"outfit")["parts"],flush=True)
    if "hair_variant.png" in item:
        WARDROBE_HAIR_PNG = str(dest/"hair_variant.png")
        Path(WARDROBE_HAIR_PNG).write_bytes(item["hair_variant.png"])
        print("[2D 헤어] 유효한 분리형 파츠:",inspect_image(WARDROBE_HAIR_PNG,"hair")["parts"],flush=True)
    else:
        print("[2D] 헤어 이미지 없음: 원본 중립 베이스에는 헤어가 없으므로 "
              "이 결과는 완성된 방송용 헤어 모델이 아닙니다.",flush=True)
elif TASK == "액세서리 제작" and ACCESSORY_SUBTYPE == "3D 교체 의상(XWear)":
    if ACCESSORY_BASE_VRM_PATH.strip():
        WARDROBE_BASE_VRM = str(Path(ACCESSORY_BASE_VRM_PATH.strip()).expanduser().resolve(strict=True))
        if not _has_vrm_container(Path(WARDROBE_BASE_VRM)):
            raise ValueError("기준 VRM이 유효하지 않습니다.")
    elif _latest_avatar():
        WARDROBE_BASE_VRM = str(_latest_avatar())
    else:
        print("[3D 의상] 원본 아바타 .vrm 파일을 업로드하세요.",flush=True)
        WARDROBE_BASE_VRM = _stored_uploads(
            files.upload, UPLOAD_ROOT/"wardrobe_base_vrm", images=False, multiple=False)[0]
        if not _has_vrm_container(Path(WARDROBE_BASE_VRM)):
            raise ValueError("기준 VRM이 유효하지 않습니다.")
    if WARDROBE_XWEAR_PATH.strip():
        WARDROBE_XWEAR = str(Path(WARDROBE_XWEAR_PATH.strip()).expanduser().resolve(strict=True))
    else:
        print("[3D 의상] VRoid Studio 호환 실제 costume.xwear를 업로드하세요. "
              "PNG 4뷰만으로 스키닝 의상이 완성되지는 않습니다.",flush=True)
        blob = files.upload()
        if set(blob) != {"costume.xwear"}:
            raise ValueError("costume.xwear 한 개만 업로드하세요.")
        dest = UPLOAD_ROOT/"xwear"
        dest.mkdir(parents=True,exist_ok=True)
        WARDROBE_XWEAR = str(dest/"costume.xwear")
        Path(WARDROBE_XWEAR).write_bytes(blob["costume.xwear"])
    print("[3D 의상] 실제 피팅/스키닝/애니메이션 검증은 VRoid 편집기에서 수행",flush=True)
elif TASK == "액세서리 제작":
    if ACCESSORY_BASE_VRM_PATH.strip():
        UPLOADED_ACCESSORY_BASE = str(Path(ACCESSORY_BASE_VRM_PATH.strip()).expanduser().resolve(strict=True))
        if not _has_vrm_container(Path(UPLOADED_ACCESSORY_BASE)):
            raise ValueError("기준 VRM 파일이 유효하지 않습니다.")
        print("[업로드] 기존 기준 VRM 사용:", UPLOADED_ACCESSORY_BASE, flush=True)
    elif not _latest_avatar():
        print("[업로드] 기준 VRM 파일 1개를 선택하세요.", flush=True)
        UPLOADED_ACCESSORY_BASE = _stored_uploads(
            files.upload, UPLOAD_ROOT / "base_vrm", images=False, multiple=False
        )[0]
        if not _has_vrm_container(Path(UPLOADED_ACCESSORY_BASE)):
            raise ValueError("업로드한 기준 VRM 파일이 유효하지 않습니다.")
    else:
        print("[업로드] 이전에 완성한 기준 VRM을 재사용합니다.", flush=True)
    print("[업로드] 액세서리 이미지 1~8장을 선택하세요.", flush=True)
    UPLOADED_ACCESSORY_IMAGES = _stored_uploads(
        files.upload, UPLOAD_ROOT / "accessories", images=True, multiple=True
    )
    if not 1 <= len(UPLOADED_ACCESSORY_IMAGES) <= 8:
        raise ValueError("액세서리 이미지는 1~8장이어야 합니다.")
    print("[업로드 완료] 액세서리 이미지:", len(UPLOADED_ACCESSORY_IMAGES), "장", flush=True)
else:
    raise ValueError("지원하지 않는 TASK: " + str(TASK))
print("업로드 셀이 끝났습니다. 다음 ⑤ 제작 셀을 수동 실행하세요.", flush=True)


## ⑨ 모델 제작


In [ ]:
#@title ⑨ 제작 전용 셀 (업로드·다운로드 없음)
#@markdown ⑧ 업로드가 끝난 뒤 실행합니다. 작업이 끝나기 전까지 이 셀을 실행 상태로 유지합니다.
import pathlib, sys, os
repo = pathlib.Path("/content/Virtual-pipeline")
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
from tools.colab_native import generate
from tools.colab_mode_ui import selection_signature
if globals().get("MODEL_DOWNLOAD_SELECTION") != selection_signature(globals()):
    raise RuntimeError("② 옵션이 ⑦ 모델 준비 이후 변경됐습니다. ⑦부터 다시 실행하세요.")

# Generation must only consume already downloaded/verified model checkpoints.
# This flag is inherited by the isolated generation worker.
os.environ["VTUBER_NOTEBOOK_EXPLICIT_DOWNLOAD"] = "1"
if "MODEL_DOWNLOAD_READY" not in globals():
    raise RuntimeError("③ AI 모델·의존성·프로그램 다운로드 셀을 먼저 실행하세요.")
from tools.colab_mode_ui import internal_scope, input_profile
selected_scope = internal_scope(TASK, MODE, LIVE2D_EDITION, ACCESSORY_SUBTYPE, OUTFIT_2D_TARGET)
if MODEL_DOWNLOAD_READY != selected_scope:
    raise RuntimeError("제작 모드를 변경했습니다. ⑦ 다운로드 셀을 다시 실행하세요.")
if TASK == "캐릭터 생성" and MODE == "live2d":
    if globals().get("MODEL_DOWNLOAD_EDITION") != LIVE2D_EDITION:
        raise RuntimeError("Live2D FREE/PRO 선택이 바뀌었습니다. ⑦ 모델 준비를 다시 실행하세요.")
    if globals().get("MODEL_DOWNLOAD_QWEN") != LIVE2D_USE_QWEN:
        raise RuntimeError("Qwen 실행 옵션이 바뀌었습니다. ⑦ 모델 준비를 다시 실행하세요.")
selected_profile = input_profile(TASK, MODE, ACCESSORY_SUBTYPE, TWO_D_INPUT, MULTI_REFERENCE_3D)
if globals().get("MODEL_DOWNLOAD_PROFILE") != selected_profile:
    raise RuntimeError("이미지 입력 방식을 변경했습니다. ③ 프로그램·모델 준비 셀을 다시 실행하세요.")
if selected_profile == "sheets":
    if not globals().get("SHEET_PACK_PATH") or not pathlib.Path(SHEET_PACK_PATH).is_file():
        raise RuntimeError("④ 셀에서 시트 ZIP 1개를 업로드하세요.")
    if MODE in ("live2d", "inochi2d"):
        os.environ["VTUBER_2D_SUPPLIED_LAYERS"] = "1"
        os.environ["VTUBER_2D_STRICT_LAYER_INPUT"] = "1"
    else:
        os.environ.pop("VTUBER_2D_SUPPLIED_LAYERS", None)
        os.environ.pop("VTUBER_2D_STRICT_LAYER_INPUT", None)
elif selected_profile == "provided_layers":
    if not globals().get("LAYER_ZIP_PATH") or not pathlib.Path(LAYER_ZIP_PATH).is_file():
        raise RuntimeError("④ 셀에서 기준 원본과 의상·헤어 없는 레이어 PNG 21장을 업로드하세요.")
    os.environ["VTUBER_2D_SUPPLIED_LAYERS"] = "1"
    os.environ["VTUBER_2D_STRICT_LAYER_INPUT"] = "1"
elif selected_profile == "wardrobe_2d":
    if (not globals().get("WARDROBE_BASE_SHEET_ZIP")
            or not globals().get("WARDROBE_OUTFIT_PNG")):
        raise RuntimeError("④ 원본 캐릭터 시트와 새 의상 outfit_variant.png가 필요합니다.")
    os.environ["VTUBER_2D_SUPPLIED_LAYERS"] = "1"
    os.environ["VTUBER_2D_STRICT_LAYER_INPUT"] = "1"
else:
    os.environ.pop("VTUBER_2D_SUPPLIED_LAYERS", None)
    os.environ.pop("VTUBER_2D_STRICT_LAYER_INPUT", None)
RESULT_FILE = None
if TASK == "캐릭터 생성" and MODE == "live2d":
    if (globals().get("LIVE2D_INPUT_MODE") != MODE or
            globals().get("LIVE2D_INPUT_EDITION") != LIVE2D_EDITION or
            globals().get("LIVE2D_INPUT_FRAMING") != LIVE2D_FRAMING or
            globals().get("LIVE2D_INPUT_ASSET") != (LIVE2D_PRO_ASSET if LIVE2D_EDITION == "pro" else None)):
        raise RuntimeError("② 모드/프레이밍 변경: ④ 새 이미지 업로드를 다시 실행하세요.")
    if not globals().get("INPUT_IMAGE_PATH") or not pathlib.Path(INPUT_IMAGE_PATH).is_file():
        raise RuntimeError("④에서 완성 캐릭터 이미지부터 준비하세요.")
    from tools.vts_production import make_cubism_handoff
    from pathlib import Path
    import uuid
    manifest = Path("/content/vtuber_builder/third_party/vts_setup_manifest.json")
    if not manifest.is_file():
        raise RuntimeError("③ VTS 모델·프로그램 준비를 먼저 완료하세요.")
    setup_info = __import__("json").loads(manifest.read_text())
    os.environ["VTUBER_SEETHROUGH_PYTHON"] = setup_info["see_through_python"]
    target = Path("/content/vtuber_builder/vts_output") / ("vts_"+uuid.uuid4().hex[:12])
    print("[VTS] See-through NF4 → 분리 PSD 제작 → Cubism Editor 입력 ZIP 시작",flush=True)
    print("[VTS] Stable-Layers + Qwen은 대형 모델 단계: 기본 See-through와 별도 실행 검증이 필요합니다.",flush=True)
    report=make_cubism_handoff(
        Path(INPUT_IMAGE_PATH),target,
        edition=LIVE2D_EDITION,scope=LIVE2D_FRAMING,
        third_party=Path("/content/vtuber_builder/third_party/see-through"),
        qwen=LIVE2D_USE_QWEN,
        asset_kind=LIVE2D_PRO_ASSET if LIVE2D_EDITION == "pro" else None,
        reference_image=(pathlib.Path(PRO_BASE_REFERENCE_PATH) if PRO_BASE_REFERENCE_PATH else None))
    RESULT_FILE=report["package"]
    print("[VTS] 레이어 PSD 제작 완료 · 최종 ZIP:",RESULT_FILE,flush=True)
    print("[VTS] PSD만 제작합니다. 리깅·물리·MOC3는 Cubism Editor에서 진행하세요.",flush=True)
elif TASK == "캐릭터 생성":
    if selected_profile != "sheets" and (not globals().get("INPUT_IMAGE_PATH") or not pathlib.Path(INPUT_IMAGE_PATH).is_file()):
        raise RuntimeError("⑧ 업로드 셀에서 캐릭터 원본을 먼저 준비하세요.")
    RESULT_FILE = generate(
        MODE, USAGE, image_path=INPUT_IMAGE_PATH, full_body=MULTI_REFERENCE_3D,
        reference_face_path=globals().get("REFERENCE_FACE_PATH", ""),
        reference_back_path=globals().get("REFERENCE_BACK_PATH", ""),
        reference_left_path=globals().get("REFERENCE_LEFT_PATH", ""),
        reference_right_path=globals().get("REFERENCE_RIGHT_PATH", ""),
        layers_zip_path=globals().get("LAYER_ZIP_PATH", ""),
        sheet_zip_path=globals().get("SHEET_PACK_PATH", "") if selected_profile == "sheets" else "",
        upload=lambda: (_ for _ in ()).throw(RuntimeError(
            "제작 중 업로드 호출 금지: ⑧ 업로드 셀을 먼저 실행하세요."
        )),
    )
elif TASK == "액세서리 제작" and ACCESSORY_SUBTYPE == "2D 교체 의상":
    import json
    from tools.modular_avatar_pack import inspect_image
    from tools.sheet_input_loader import inspect_sheet_archive
    source = globals().get("WARDROBE_BASE_SHEET_ZIP","")
    outfit = globals().get("WARDROBE_OUTFIT_PNG","")
    if not source or not outfit or not pathlib.Path(source).is_file() or not pathlib.Path(outfit).is_file():
        raise RuntimeError("④에서 중립 베이스 시트 ZIP과 outfit_variant.png를 업로드하세요.")
    inspect_sheet_archive(source,"live2d")
    inspect_image(outfit,"outfit")
    hair = globals().get("WARDROBE_HAIR_PNG","")
    if hair:
        if not pathlib.Path(hair).is_file():
            raise RuntimeError("헤어 파일이 없습니다: "+hair)
        inspect_image(hair,"hair")
    wardrobe_dir = pathlib.Path("/content/vtuber_builder/wardrobe_output")
    wardrobe_dir.mkdir(parents=True,exist_ok=True)
    request_path = wardrobe_dir/"wardrobe_gpu_request.json"
    result_path = wardrobe_dir/"wardrobe_gpu_result.json"
    result_path.unlink(missing_ok=True)
    request_path.write_text(json.dumps({
        "mode":"wardrobe_2d", "sheet_zip":str(pathlib.Path(source).resolve()),
        "outfit_png":str(pathlib.Path(outfit).resolve()),
        "hair_png":str(pathlib.Path(hair).resolve()) if hair else None,
        "output_dir":str(wardrobe_dir/"rendered"),
    }),encoding="utf-8")
    # This is a disposable GPU process, so SR model memory is released
    # before the actual 2D rig/editor generation begins.
    run([
        sys.executable,"-u",str(repo/"tools"/"sheet_prepare_worker.py"),
        "--request",str(request_path),"--result",str(result_path),
    ],timeout=7200)
    if not result_path.is_file():
        raise RuntimeError("의상 파츠 처리 작업이 결과 manifest를 내지 않았습니다.")
    generated = json.loads(result_path.read_text(encoding="utf-8"))
    if not all(pathlib.Path(generated[k]).is_file() for k in ("front","layers")):
        raise RuntimeError("베이스/의상/헤어 제작 자료가 생성되지 않았습니다.")
    print("[2D] 중립 베이스 20개 + 의상 4개 + "
          f"헤어 {4 if hair else 0}개 = {24+(4 if hair else 0)}개 리깅",
          flush=True)
    RESULT_FILE = generate(
        OUTFIT_2D_TARGET, USAGE,
        image_path=generated["front"], layers_zip_path=generated["layers"],
        upload=lambda: (_ for _ in ()).throw(RuntimeError(
            "제작 중 업로드 호출 금지: ④에서 자료를 먼저 업로드하세요."
        )),
    )
    print("[2D 의상] 파츠 출력:",generated["layers"],flush=True)
    print("[2D 의상] 실시간 코스튬 스위치/슬라이더는 "
          "각 Cubism/Inochi 편집기에서 키 연결 확인이 필요합니다.",flush=True)
elif TASK == "액세서리 제작" and ACCESSORY_SUBTYPE == "3D 교체 의상(XWear)":
    from tools.wardrobe_handoff import prepare_vroid_dressup
    source = globals().get("WARDROBE_BASE_VRM","")
    xwear = globals().get("WARDROBE_XWEAR","")
    if not source or not xwear or not pathlib.Path(source).is_file() or not pathlib.Path(xwear).is_file():
        raise RuntimeError("④에서 유효한 원본 VRM과 costume.xwear를 모두 지정하세요.")
    output_root = pathlib.Path("/content/vtuber_builder/wardrobe_output")
    output_root.mkdir(parents=True,exist_ok=True)
    RESULT_FILE = prepare_vroid_dressup(
        source,xwear,str(output_root/"vroid_dressup_handoff.zip"))
    print("[3D 의상] VRoid 편집기 전달 ZIP 준비 완료. "
          "착용 완료 VRM이 아닙니다. 공식 피팅·메시 삭제·모션 검증 후 VRM을 내보내세요.",
          flush=True)
elif TASK == "액세서리 제작":
    accessories = globals().get("UPLOADED_ACCESSORY_IMAGES", [])
    if not accessories or any(not pathlib.Path(x).is_file() for x in accessories):
        raise RuntimeError("⑧ 업로드 셀에서 액세서리 이미지를 먼저 준비하세요.")
    RESULT_FILE = generate(
        "accessory", USAGE, accessory_anchor=ACCESSORY_ANCHOR,
        accessory_base_path=globals().get("UPLOADED_ACCESSORY_BASE", ""),
        accessory_image_paths=tuple(accessories),
        upload=lambda: (_ for _ in ()).throw(RuntimeError(
            "제작 중 업로드 호출 금지: ⑧ 업로드 셀을 먼저 실행하세요."
        )),
    )
else:
    raise ValueError("작업 종류가 잘못됐습니다: " + str(TASK))

if not RESULT_FILE:
    raise RuntimeError(
        "제작 실패: 모델 작업이 완료되지 않았습니다. 위의 단계별 실패 원인과 "
        "/content/vtuber_builder/jobs/*/generation.log를 확인하세요."
    )
RESULT_FILE = str(pathlib.Path(RESULT_FILE).resolve())
if not pathlib.Path(RESULT_FILE).is_file():
    raise RuntimeError("제작 결과 파일이 존재하지 않습니다: " + RESULT_FILE)
print("Cubism 편집기 전달 자료 준비 완료 (MOC3 아님):" if MODE == "live2d" and TASK == "캐릭터 생성" else "제작 완료, 결과 파일:", RESULT_FILE, flush=True)
print("다운로드는 ⑥ 결과 다운로드 셀에서 따로 실행하세요.", flush=True)


## ⑩ 결과 파일 다운로드


In [ ]:
#@title ⑩ 제작 결과 파일 다운로드 전용 셀 (완성 후 수동 실행)
#@markdown 제작이 성공했을 때만 다운로드를 누르세요.
DOWNLOAD_NOW = False #@param {type:"boolean"}
from pathlib import Path
if not DOWNLOAD_NOW:
    print("다운로드 대기 중. 다운로드하려면 DOWNLOAD_NOW를 켜고 ⑩ 셀을 다시 실행하세요.")
else:
    result = globals().get("RESULT_FILE")
    if not result or not Path(str(result)).is_file():
        raise RuntimeError("검증된 제작 결과가 없습니다. ⑤ 제작 셀을 먼저 완료하세요.")
    from google.colab import files
    print("다운로드 시작:", result, flush=True)
    files.download(str(result))


## ⑪ 상태 및 로그 진단


In [ ]:
#@title ⑪ 상태 진단 셀 · 실제 제작 상태 및 실패 로그 확인 (재연결 후에도 수동 실행 가능)
#@markdown 이 셀은 **모델 생성 완료를 자동으로 주장하지 않습니다.** 파일과 상태 기록을 검증합니다.
import json
from pathlib import Path

jobs = Path("/content/vtuber_builder/jobs")
reports = sorted(jobs.glob("*/status.json"), key=lambda x: x.stat().st_mtime, reverse=True) if jobs.is_dir() else []
result_file = globals().get("RESULT_FILE")
if reports:
    status_file = reports[0]
    data = json.loads(status_file.read_text(encoding="utf-8"))
    state = str(data.get("state", "unknown"))
    job_folder = status_file.parent
    print("최근 작업:", job_folder, flush=True)
    print("실제 작업 상태:", state, flush=True)
    print("worker PID:", data.get("pid"), "종료 코드:", data.get("exit_code", "기록 없음"), flush=True)
    print("전체 로그:", job_folder / "generation.log", flush=True)
    if state == "complete":
        if not result_file or not Path(str(result_file)).is_file():
            print("주의: 상태 기록은 complete이지만, 현재 세션의 결과 파일이 확인되지 않았습니다.", flush=True)
    elif state == "running":
        print("주의: running 기록만으로는 실제 프로세스가 살아 있다고 확인할 수 없습니다.", flush=True)
    else:
        print("제작 실패/취소 · 완성 모델로 취급하지 않습니다.", flush=True)
elif result_file and Path(str(result_file)).is_file():
    print("검증된 결과 파일:", Path(str(result_file)).resolve(), flush=True)
else:
    print("검증된 결과 파일이 없습니다. 작업 중 Colab 런타임이 삭제되면 /content 의 임시 로그도 사라질 수 있습니다.", flush=True)
print("Colab 오른쪽에 '런타임에 연결되어 있지 않습니다'가 표시된다면 이 셀도 실행할 수 없습니다.", flush=True)
print("먼저 Colab '다시 연결'을 시도해야 합니다. 무료 T4 할당량이 없다면 런타임이 재할당될 때까지 재실행할 수 없습니다.", flush=True)

# VTS 전용 결과는 모델 완성이 아닌 Cubism Editor handoff입니다.
vts_reports = sorted(Path("/content/vtuber_builder/vts_output").glob("*/vts_status.json"), key=lambda x:x.stat().st_mtime, reverse=True)
if vts_reports:
    vts_data=json.loads(vts_reports[0].read_text())
    print("[VTS 최근 작업]",vts_reports[0],"state=",vts_data.get("state"),"moc3_generated=",vts_data.get("moc3_generated"))


## ⑫ 마지막 셀


In [ ]:
#@title ⑫ 마지막 셀 (작업 수행 없음)
print("마지막 셀: ⑦ 프로그램·모델 다운로드 / ⑧ 사진 업로드 / ⑨ 제작 / ⑩ 결과 파일 다운로드")
print("Colab이 연결을 끊은 경우 마지막 셀만으로 런타임을 유지할 수는 없습니다.")
